In [25]:
!pip install openpyxl -q

In [26]:
import logging
import sqlite3
from dataclasses import dataclass, field
from datetime import datetime

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("pipeline")

INPUT_PATH = "/content/Python_Data_Pipeline_Lab_Dataset (1).xlsx"
OUTPUT_DB = "retail_dw.db"

**Task 1 - Pipeline Configuration และ Extract**

In [27]:
from dataclasses import dataclass, field

@dataclass
class PipelineConfig:
    input_path: str
    output_db: str
    batches: list = field(default_factory=lambda: ["orders_batch_1", "orders_batch_2", "orders_batch_3"])
    error_mode: str = "quarantine"  # "quarantine" or "fail_fast"


def extract_dimension(config: PipelineConfig, sheet_name: str) -> pd.DataFrame:
    logger.info(f"Extracting dimension sheet: {sheet_name}")
    df = pd.read_excel(config.input_path, sheet_name=sheet_name)
    logger.info(f"  -> {len(df)} rows read")
    return df


def extract_batch(config: PipelineConfig, batch_name: str):
    started_at = datetime.now()
    try:
        df = pd.read_excel(config.input_path, sheet_name=batch_name)
        ended_at = datetime.now()
        logger.info(
            f"Batch '{batch_name}': read {len(df)} rows "
            f"(started={started_at.isoformat()}, ended={ended_at.isoformat()})"
        )
        return df, started_at, ended_at
    except Exception as e:
        ended_at = datetime.now()
        logger.error(f"Batch '{batch_name}' FAILED to read: {e}")
        return None, started_at, ended_at

**Task 2 - Transform และ Data Quality**

In [28]:
PAYMENT_METHOD_MAP = {
    "cash": "Cash",
    "credit card": "Credit Card",
    "promptpay": "PromptPay",
    "bank transfer": "Bank Transfer",
}
SALES_CHANNEL_MAP = {
    "e-commerce": "Online",  # data_dictionary rule: map E-Commerce -> Online
}

In [29]:
def normalize_categorical(series: pd.Series, mapping: dict) -> pd.Series:
    lowered = series.astype(str).str.strip().str.lower()
    mapped = lowered.map(mapping)
    return mapped.fillna(series.astype(str).str.strip())


def transform_batch(orders_df: pd.DataFrame, customers_df: pd.DataFrame,
                     products_df: pd.DataFrame, source_batch: str):
    """Clean + validate one batch. Returns (clean_df, quarantine_df)."""
    df = orders_df.copy()
    reasons = pd.Series([[] for _ in range(len(df))], index=df.index)

    def add_reason(mask: pd.Series, code: str):
        for idx in df.index[mask]:
            reasons.at[idx].append(code)

    # 1) Safe type conversion
    df["order_datetime"] = pd.to_datetime(df["order_datetime"], errors="coerce")
    df["updated_at"] = pd.to_datetime(df["updated_at"], errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
    df["discount_pct"] = pd.to_numeric(df["discount_pct"], errors="coerce")

    add_reason(df["order_datetime"].isna(), "INVALID_DATETIME")
    add_reason(df["quantity"].isna(), "INVALID_QUANTITY_TYPE")
    add_reason(df["unit_price"].isna(), "INVALID_PRICE_TYPE")

    # 2) Normalize categorical values
    df["payment_method"] = normalize_categorical(df["payment_method"], PAYMENT_METHOD_MAP)
    df["sales_channel"] = normalize_categorical(df["sales_channel"], SALES_CHANNEL_MAP)

    # 3) Business rule validation
    bad_qty = ~df["quantity"].between(1, 20, inclusive="both").fillna(False)
    bad_price = ~(df["unit_price"] > 0).fillna(False)
    bad_discount = ~df["discount_pct"].between(0, 100, inclusive="both").fillna(False)
    add_reason(bad_qty, "INVALID_QUANTITY_RANGE")
    add_reason(bad_price, "INVALID_PRICE_RANGE")
    add_reason(bad_discount, "INVALID_DISCOUNT_RANGE")

    # 4) Referential integrity
    valid_customer = df["customer_id"].isin(customers_df["customer_id"])
    valid_product = df["product_id"].isin(products_df["product_id"])
    add_reason(~valid_customer, "FK_CUSTOMER_NOT_FOUND")
    add_reason(~valid_product, "FK_PRODUCT_NOT_FOUND")

    # 5) Deduplicate by order_id, keep latest updated_at
    df["_is_duplicate"] = df.duplicated(subset="order_id", keep=False)
    df_sorted = df.sort_values("updated_at")
    keep_idx = df_sorted.drop_duplicates(subset="order_id", keep="last").index
    is_dropped_dup = pd.Series(~df.index.isin(keep_idx), index=df.index)
    add_reason(df["_is_duplicate"] & is_dropped_dup, "DUPLICATE_OLD_VERSION")

    # 6) Derived columns
    df["gross_amount"] = df["quantity"] * df["unit_price"]
    df["net_amount"] = df["gross_amount"] * (1 - df["discount_pct"] / 100)

    # 7) Split clean vs quarantine
    df["reason_code"] = reasons.apply(lambda lst: ";".join(lst) if lst else "")
    df["source_batch"] = source_batch
    has_reason = df["reason_code"] != ""
    is_bad = has_reason | is_dropped_dup

    quarantine_df = df[is_bad].copy()
    dup_in_quarantine = is_dropped_dup.loc[quarantine_df.index]
    quarantine_df.loc[
        dup_in_quarantine & (quarantine_df["reason_code"] == ""), "reason_code"
    ] = "DUPLICATE_OLD_VERSION"

    clean_df = df[~is_bad].drop(columns=["_is_duplicate", "reason_code"])

    logger.info(f"Transform '{source_batch}': {len(df)} read -> {len(clean_df)} valid, {len(quarantine_df)} quarantined")
    return clean_df, quarantine_df

**Task 3 - Star Schema และ Load**

In [30]:
DDL = """
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id  TEXT UNIQUE NOT NULL,
    customer_name TEXT,
    province TEXT,
    segment TEXT
);

CREATE TABLE IF NOT EXISTS dim_product (
    product_key INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id  TEXT UNIQUE NOT NULL,
    product_name TEXT,
    category TEXT
);

CREATE TABLE IF NOT EXISTS dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT UNIQUE NOT NULL,
    day INTEGER,
    month INTEGER,
    quarter INTEGER,
    year INTEGER
);

-- Grain: one validated order-product line item per row
CREATE TABLE IF NOT EXISTS fact_sales (
    fact_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id TEXT NOT NULL,
    date_key INTEGER,
    customer_key INTEGER,
    product_key INTEGER,
    quantity INTEGER,
    unit_price REAL,
    discount_pct REAL,
    gross_amount REAL,
    net_amount REAL,
    payment_method TEXT,
    sales_channel TEXT,
    UNIQUE(order_id, product_key),
    FOREIGN KEY(customer_key) REFERENCES dim_customer(customer_key),
    FOREIGN KEY(product_key) REFERENCES dim_product(product_key)
);

CREATE TABLE IF NOT EXISTS pipeline_run_log (
    run_id INTEGER PRIMARY KEY AUTOINCREMENT,
    batch TEXT,
    started_at TEXT,
    ended_at TEXT,
    rows_read INTEGER,
    rows_valid INTEGER,
    rows_rejected INTEGER,
    rows_loaded INTEGER,
    fact_count_after INTEGER,
    status TEXT
);
"""

In [31]:
def init_db(config: PipelineConfig):
    conn = sqlite3.connect(config.output_db)
    conn.executescript(DDL)
    conn.commit()
    return conn


def load_dimensions(conn, customers_df: pd.DataFrame, products_df: pd.DataFrame):
    cur = conn.cursor()
    for _, row in customers_df.iterrows():
        cur.execute(
            "INSERT OR IGNORE INTO dim_customer (customer_id, customer_name, province, segment) VALUES (?,?,?,?)",
            (row["customer_id"], row["customer_name"], row["province"], row["segment"]),
        )
    for _, row in products_df.iterrows():
        cur.execute(
            "INSERT OR IGNORE INTO dim_product (product_id, product_name, category) VALUES (?,?,?)",
            (row["product_id"], row["product_name"], row["category"]),
        )
    conn.commit()


def get_or_create_date_key(cur, dt: pd.Timestamp) -> int:
    date_key = int(dt.strftime("%Y%m%d"))
    cur.execute(
        "INSERT OR IGNORE INTO dim_date (date_key, full_date, day, month, quarter, year) VALUES (?,?,?,?,?,?)",
        (date_key, dt.strftime("%Y-%m-%d"), dt.day, dt.month, (dt.month - 1) // 3 + 1, dt.year),
    )
    return date_key


def load_facts(conn, clean_df: pd.DataFrame) -> int:
    """Upsert into fact_sales. UNIQUE(order_id, product_key) makes this idempotent;
    ON CONFLICT DO UPDATE lets a newer updated_at overwrite an older loaded row."""
    cur = conn.cursor()
    loaded = 0
    conn.execute("BEGIN")
    try:
        for _, row in clean_df.iterrows():
            cust_key = cur.execute(
                "SELECT customer_key FROM dim_customer WHERE customer_id=?", (row["customer_id"],)
            ).fetchone()[0]
            prod_key = cur.execute(
                "SELECT product_key FROM dim_product WHERE product_id=?", (row["product_id"],)
            ).fetchone()[0]
            date_key = get_or_create_date_key(cur, row["order_datetime"])

            cur.execute(
                """
                INSERT INTO fact_sales
                    (order_id, date_key, customer_key, product_key, quantity, unit_price,
                     discount_pct, gross_amount, net_amount, payment_method, sales_channel)
                VALUES (?,?,?,?,?,?,?,?,?,?,?)
                ON CONFLICT(order_id, product_key) DO UPDATE SET
                    quantity=excluded.quantity,
                    unit_price=excluded.unit_price,
                    discount_pct=excluded.discount_pct,
                    gross_amount=excluded.gross_amount,
                    net_amount=excluded.net_amount,
                    payment_method=excluded.payment_method,
                    sales_channel=excluded.sales_channel
                """,
                (
                    row["order_id"], date_key, cust_key, prod_key,
                    int(row["quantity"]), float(row["unit_price"]), float(row["discount_pct"]),
                    float(row["gross_amount"]), float(row["net_amount"]),
                    row["payment_method"], row["sales_channel"],
                ),
            )
            loaded += 1
        conn.commit()
    except Exception as e:
        conn.rollback()
        logger.error(f"Load failed, rolled back: {e}")
        raise
    return loaded

**Task 4 - Idempotency และ Incremental Loading**

In [32]:
def log_run(conn, batch, started_at, ended_at, rows_read, rows_valid, rows_rejected, rows_loaded, status):
    fact_count_after = conn.execute("SELECT COUNT(*) FROM fact_sales").fetchone()[0]
    conn.execute(
        """INSERT INTO pipeline_run_log
           (batch, started_at, ended_at, rows_read, rows_valid, rows_rejected,
            rows_loaded, fact_count_after, status)
           VALUES (?,?,?,?,?,?,?,?,?)""",
        (batch, started_at.isoformat(), ended_at.isoformat(),
         rows_read, rows_valid, rows_rejected, rows_loaded, fact_count_after, status),
    )
    conn.commit()
    logger.info(f"[run_log] batch={batch} status={status} fact_count_after={fact_count_after}")


def run_batch(conn, config: PipelineConfig, customers_df, products_df, batch_name, all_quarantine, kpi):
    """Process a single batch end-to-end: extract -> transform -> validate -> load."""
    raw_df, started_at, ended_at = extract_batch(config, batch_name)

    if raw_df is None:
        log_run(conn, batch_name, started_at, ended_at, 0, 0, 0, 0, "failed")
        kpi["batches_failed"] += 1
        return

    try:
        clean_df, quarantine_df = transform_batch(raw_df, customers_df, products_df, batch_name)
        loaded = load_facts(conn, clean_df)
        all_quarantine.append(quarantine_df)

        kpi["rows_read"] += len(raw_df)
        kpi["rows_valid"] += len(clean_df)
        kpi["rows_rejected"] += len(quarantine_df)
        kpi["rows_loaded"] += loaded
        kpi["net_amount_total"] += clean_df["net_amount"].sum()

        log_run(
            conn, batch_name, started_at, datetime.now(),
            rows_read=len(raw_df), rows_valid=len(clean_df),
            rows_rejected=len(quarantine_df), rows_loaded=loaded, status="success",
        )
    except Exception as e:
        # A batch-level failure does not roll back or delete rows already
        # committed from earlier batches - each batch's transaction is independent.
        logger.error(f"Batch {batch_name} failed during transform/load: {e}")
        log_run(conn, batch_name, started_at, datetime.now(), len(raw_df), 0, 0, 0, "failed")
        kpi["batches_failed"] += 1


def run_pipeline(config: PipelineConfig, conn=None, all_quarantine=None, kpi=None):
    """
    Orchestrator: extract -> transform -> validate -> load for every batch in config.
    Reuses an open connection/accumulators across calls when passed in, so callers
    can run this repeatedly against the same database to demonstrate idempotency.
    """
    own_conn = conn is None
    if own_conn:
        conn = init_db(config)
    customers_df = extract_dimension(config, "customers")
    products_df = extract_dimension(config, "products")
    load_dimensions(conn, customers_df, products_df)

    if all_quarantine is None:
        all_quarantine = []
    if kpi is None:
        kpi = {"rows_read": 0, "rows_valid": 0, "rows_rejected": 0, "rows_loaded": 0,
               "net_amount_total": 0.0, "batches_failed": 0}

    for batch_name in config.batches:
        run_batch(conn, config, customers_df, products_df, batch_name, all_quarantine, kpi)

    if own_conn:
        conn.close()

    return all_quarantine, kpi

In [33]:
def main():
    import os
    if os.path.exists(OUTPUT_DB):
        os.remove(OUTPUT_DB)  # fresh run for a clean demonstration

    config = PipelineConfig(input_path=INPUT_PATH, output_db=OUTPUT_DB)
    conn = init_db(config)
    all_quarantine, kpi = [], {
        "rows_read": 0, "rows_valid": 0, "rows_rejected": 0, "rows_loaded": 0,
        "net_amount_total": 0.0, "batches_failed": 0,
    }

    # Round 1: batch_1 first run
    logger.info("=== ROUND 1: orders_batch_1 (first run) ===")
    run_pipeline(PipelineConfig(INPUT_PATH, OUTPUT_DB, batches=["orders_batch_1"]),
                 conn=conn, all_quarantine=all_quarantine, kpi=kpi)
    fact_after_round1 = conn.execute("SELECT COUNT(*) FROM fact_sales").fetchone()[0]

    # Round 2: batch_1 rerun - proves idempotency (fact count must not grow)
    logger.info("=== ROUND 2: orders_batch_1 (re-run, idempotency check) ===")
    run_pipeline(PipelineConfig(INPUT_PATH, OUTPUT_DB, batches=["orders_batch_1"]),
                 conn=conn, all_quarantine=all_quarantine, kpi=kpi)
    fact_after_round2 = conn.execute("SELECT COUNT(*) FROM fact_sales").fetchone()[0]
    assert fact_after_round1 == fact_after_round2, "Idempotency check FAILED: fact count changed on re-run!"
    logger.info(f"Idempotency check PASSED: fact_sales stayed at {fact_after_round2} rows after re-run")

    # Round 3: batch_2 (incremental)
    logger.info("=== ROUND 3: orders_batch_2 ===")
    run_pipeline(PipelineConfig(INPUT_PATH, OUTPUT_DB, batches=["orders_batch_2"]),
                 conn=conn, all_quarantine=all_quarantine, kpi=kpi)

    # Round 4: batch_3 (incremental)
    logger.info("=== ROUND 4: orders_batch_3 ===")
    run_pipeline(PipelineConfig(INPUT_PATH, OUTPUT_DB, batches=["orders_batch_3"]),
                 conn=conn, all_quarantine=all_quarantine, kpi=kpi)

    # Write quarantine.csv
    if all_quarantine:
        pd.concat(all_quarantine, ignore_index=True).to_csv("quarantine.csv", index=False)

    # Write pipeline_run_log.csv (mirrors the SQLite table)
    run_log_df = pd.read_sql("SELECT * FROM pipeline_run_log", conn)
    run_log_df.to_csv("pipeline_run_log.csv", index=False)

    fact_count = conn.execute("SELECT COUNT(*) FROM fact_sales").fetchone()[0]
    logger.info(
        "=== KPI SUMMARY === "
        f"rows_read={kpi['rows_read']} valid={kpi['rows_valid']} "
        f"rejected={kpi['rows_rejected']} loaded={kpi['rows_loaded']} "
        f"fact_sales_final={fact_count} net_amount_total={kpi['net_amount_total']:,.2f}"
    )

    conn.close()
    return kpi, fact_count


if __name__ == "__main__":
    main()